# Notebook 07c — 2D Encoder Without L2 Normalization (Option B)
**Project:** ENSO-BSISO Self-Supervised Learning  
**Author:** Jiayi (jh9141@nyu.edu)

## Why this notebook exists

Notebook 07b (temperature sweep) showed that even when training converges (full angular spread on the unit circle), **BSISO phase probe plateaus at ~33%** across all τ — far below the 64D baseline of 67.7%.

**Diagnosis:** L2-normalizing a 2D output forces embeddings onto S¹ in R², which is a **1-dimensional manifold** (only the angle θ matters; the radius is fixed at 1). So our prior "2D" experiment was actually testing *"is BSISO 1D-on-a-circle?"*, not *"is BSISO 2D in R²?"*

**Fix:** drop L2 normalization. Embeddings live in R² freely — both angle (θ) AND radius (r) can encode information. This is the clean test of whether 2D is *truly* sufficient.

## Design choice — Variant 2: raw dot product in the loss

Two ways to drop L2 norm:
- **Variant 1**: encoder outputs raw 2D, but loss internally L2-normalizes (cosine sim). Stable, but radius gets no gradient signal → cannot encode anything systematic. Doesn't actually test the hypothesis.
- **Variant 2 (this notebook)**: encoder outputs raw 2D AND loss uses raw dot product. Both angle and radius affect similarity, so optimization can use the radius if useful. Add small weight decay (1e-4) for stability.

Variant 2 is the proper test of "does freeing the radius help?"

## What gets produced

- `checkpoints/encoder_2d_lee_no_l2_final.pth`
- `checkpoints/training_history_2d_lee_no_l2.json`
- `results/lee_2d_no_l2/`
  - `embeddings.npy` — (6579, 2), NOT L2-normalized
  - `training_curves.png` — train/val InfoNCE loss + norm trajectory
  - `embedding_2d_overview.png` — raw-scale scatter colored by phase, ENSO, and amplitude
  - `radius_diagnostics.png` — radius vs amplitude, radius distribution by phase + ENSO
  - `linear_probe_results.json` — full-2D probe
  - `angle_only_vs_full_probes.json` — comparison: probing on angle only vs raw 2D
  - `enso_displacement.png`
  - `phase1_option_b_summary.md` — decision

## Hyperparameters

| Knob | Value | Rationale |
|---|---|---|
| Embedding dim | 2 | Same as 07/07b |
| L2 normalization | **OFF** | The change being tested |
| Loss | raw dot product InfoNCE | Variant 2 |
| Temperature τ | 0.5 | Best from 07b sweep |
| Weight decay | 1e-4 | Prevent norm explosion |
| Optimizer | Adam, lr=1e-3, cosine→1e-5 | Same as 07/07b |
| Epochs | 50 | Same as 07/07b |
| Gradient clip | 1.0 | Same as 07/07b |

## Decision rule

- **Greenlight Phase 2**: phase val ≥ 62% AND z ≥ 3.0 → BSISO is genuinely 2D in R². Use this τ + no-L2 setup for SSL temporal model.
- **Radius didn't help**: phase val still ~33–40%, angle-only probe ≈ full-2D probe → BSISO is 1D under standard contrastive even when freed. Escalate to Phase 4 dim sweep.
- **Norm explosion**: max embedding norm > 100 or training diverges → Variant 2 is unstable; rerun with larger weight_decay (e.g. 1e-3) or fall back to Variant 1.

## Runtime

~30 min on T4 (single training run).

---
## Before running
1. Notebook 07 + 07b should have completed (baseline numbers come from their outputs).
2. Set runtime to T4 GPU.

## Cell 1 — Mount Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
import matplotlib.pyplot as plt

# ── Option B config ──────────────────────────────────────────────────────────
EMBEDDING_DIM = 2
APPROACH      = 'lee'
RUN_TAG       = '2d_lee_no_l2'
TEMPERATURE   = 0.5         # best from 07b sweep
EPOCHS        = 50
BATCH_SIZE    = 64
LR            = 1e-3
WEIGHT_DECAY  = 1e-4        # NEW vs notebook 07/07b — prevent norm explosion
# ─────────────────────────────────────────────────────────────────────────────

PROJECT_DIR    = '/content/drive/MyDrive/BSISO_SSL_Project'
PROCESSED_DIR  = f'{PROJECT_DIR}/data/processed'
CHECKPOINT_DIR = f'{PROJECT_DIR}/checkpoints'
RESULTS_DIR    = f'{PROJECT_DIR}/results/lee_2d_no_l2'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

X_FILE      = 'X_MJJAS_lee.npy'
LABELS_FILE = 'labels_aligned_mjjas_lee.csv'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:        {device}')
print(f'Embedding dim: {EMBEDDING_DIM}  (no L2 normalization)')
print(f'Loss:          raw dot product InfoNCE  (τ={TEMPERATURE})')
print(f'Weight decay:  {WEIGHT_DECAY}')
print(f'Run tag:       {RUN_TAG}')
print(f'Results dir:   results/lee_2d_no_l2/')

## Cell 2 — Load Data + Year-Based Split + Phase-ENSO Index

Identical to notebooks 07 / 07b.

In [ ]:
X      = np.load(f'{PROCESSED_DIR}/{X_FILE}')
labels = pd.read_csv(f'{PROCESSED_DIR}/{LABELS_FILE}', parse_dates=['date'])

print(f'X shape:  {X.shape}  (N, channels, lat, lon)')
print(f'Labels:   {len(labels)} rows')

all_years = sorted(labels['date'].dt.year.unique())
val_years = all_years[::5]
train_years = [y for y in all_years if y not in val_years]
year_col  = labels['date'].dt.year
train_idx = labels.index[year_col.isin(train_years)].values
val_idx   = labels.index[year_col.isin(val_years)].values

print(f'\nVal years ({len(val_years)}): {val_years}')
print(f'Train: {len(train_idx)} samples ({100*len(train_idx)/len(labels):.1f}%)')
print(f'Val:   {len(val_idx)} samples ({100*len(val_idx)/len(labels):.1f}%)')

phase_enso_index = defaultdict(list)
for idx in train_idx:
    row = labels.loc[idx]
    if row['bsiso_amplitude'] > 1.0:
        key = (row['bsiso_phase'], row['enso_category'])
        phase_enso_index[key].append(idx)

active_total = sum(len(v) for v in phase_enso_index.values())
print(f'\nActive BSISO days in TRAIN (amplitude > 1.0): {active_total} / {len(train_idx)}')

## Cell 3 — PairSampler + Dataset + DataLoaders

Identical to notebook 07b.

In [ ]:
class PairSampler:
    def __init__(self, labels_df, phase_enso_index):
        self.labels = labels_df
        self.index  = phase_enso_index
        self.enso_categories = labels_df['enso_category'].unique().tolist()

    def sample_positive_pair(self):
        key = self._random_category()
        indices = self.index[key]
        if len(indices) < 2:
            return self.sample_easy_negative_pair()
        idx_A, idx_B = np.random.choice(indices, size=2, replace=False)
        year_A = self.labels.loc[idx_A, 'date'].year
        other = [i for i in indices if self.labels.loc[i, 'date'].year != year_A]
        if other:
            idx_B = np.random.choice(other)
        return idx_A, idx_B, 'positive'

    def sample_hard_negative_pair(self):
        if len(self.enso_categories) < 2:
            return self.sample_easy_negative_pair()
        phase = np.random.choice(range(1, 9))
        enso_A, enso_B = np.random.choice(self.enso_categories, size=2, replace=False)
        key_A, key_B = (phase, enso_A), (phase, enso_B)
        if not self.index[key_A] or not self.index[key_B]:
            return self.sample_positive_pair()
        idx_A = np.random.choice(self.index[key_A])
        idx_B = np.random.choice(self.index[key_B])
        return idx_A, idx_B, 'hard_negative'

    def sample_easy_negative_pair(self):
        phase_A, phase_B = np.random.choice(range(1, 9), size=2, replace=False)
        enso_A = np.random.choice(self.enso_categories)
        enso_B = np.random.choice(self.enso_categories)
        key_A, key_B = (phase_A, enso_A), (phase_B, enso_B)
        idx_A = (np.random.choice(self.index[key_A]) if self.index[key_A]
                 else self.labels[self.labels['bsiso_phase'] == phase_A].sample(1).index[0])
        idx_B = (np.random.choice(self.index[key_B]) if self.index[key_B]
                 else self.labels[self.labels['bsiso_phase'] == phase_B].sample(1).index[0])
        return idx_A, idx_B, 'easy_negative'

    def _random_category(self):
        valid = [k for k in self.index if len(self.index[k]) > 0]
        return valid[np.random.randint(len(valid))]


class BSISOPairDataset(Dataset):
    def __init__(self, X_data, labels_df, phase_enso_index,
                 mode='train', train_indices=None):
        self.X       = X_data
        self.labels  = labels_df
        self.sampler = PairSampler(labels_df, phase_enso_index)
        self.mode    = mode
        self.train_indices = (train_indices if train_indices is not None
                              else np.arange(len(X_data)))
        if mode == 'val':
            self.val_pairs = self._create_val_pairs()

    def _create_val_pairs(self):
        pairs = []
        val_labels = self.labels.loc[self.train_indices]
        for phase in range(1, 9):
            for enso in self.sampler.enso_categories:
                group = val_labels[
                    (val_labels['bsiso_phase'] == phase) &
                    (val_labels['enso_category'] == enso)
                ].index.tolist()
                for i in range(len(group)):
                    for j in range(i + 1, len(group)):
                        pairs.append((group[i], group[j], 'positive'))
        return pairs[:1000]

    def __len__(self):
        return (len(self.train_indices) if self.mode == 'train'
                else len(self.val_pairs))

    def __getitem__(self, idx):
        if self.mode == 'train':
            r = np.random.rand()
            if r < 0.30:
                idx_A, idx_B, _ = self.sampler.sample_positive_pair()
            elif r < 0.50:
                idx_A, idx_B, _ = self.sampler.sample_hard_negative_pair()
            else:
                idx_A, idx_B, _ = self.sampler.sample_easy_negative_pair()
        else:
            idx_A, idx_B, _ = self.val_pairs[idx]
        field_A = torch.from_numpy(self.X[idx_A]).float()
        field_B = torch.from_numpy(self.X[idx_B]).float()
        return field_A, field_B


train_dataset = BSISOPairDataset(X, labels, phase_enso_index, mode='train', train_indices=train_idx)
val_dataset   = BSISOPairDataset(X, labels, phase_enso_index, mode='val',   train_indices=val_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train dataset: {len(train_dataset)} items  →  {len(train_loader)} batches/epoch')
print(f'Val dataset:   {len(val_dataset)} pairs   →  {len(val_loader)} batches')

## Cell 4 — CNN Encoder (NO L2 Normalization)

**Single change vs notebook 07/07b**: the final `F.normalize(z, p=2, dim=1)` line is removed. Output is raw 2D, can have any magnitude.

In [ ]:
class CNNEncoderNoL2(nn.Module):
    def __init__(self, embedding_dim=2):
        super().__init__()
        self.conv1 = nn.Conv2d(3,  32, kernel_size=3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False)
        self.bn3   = nn.BatchNorm2d(128)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc          = nn.Linear(128, embedding_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias,   0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.global_pool(x).view(x.size(0), -1)
        z = self.fc(x)
        # NOTE: no F.normalize() — raw 2D output, can have any magnitude
        return z


def InfoNCE_loss_raw(z_A, z_B, temperature):
    """Raw dot product InfoNCE — both angle AND magnitude affect similarity.
    With L2-normalized inputs this would equal cosine InfoNCE; here we let the
    encoder choose the embedding scale."""
    sim_matrix = torch.matmul(z_A, z_B.T) / temperature
    labels_ = torch.arange(z_A.size(0), device=z_A.device)
    return F.cross_entropy(sim_matrix, labels_)


# Quick test
encoder = CNNEncoderNoL2(embedding_dim=EMBEDDING_DIM)
total_params = sum(p.numel() for p in encoder.parameters())
print(f'Total parameters: {total_params:,}')

dummy = torch.randn(4, 3, 31, 51)
out   = encoder(dummy)
print(f'\nInput:  {dummy.shape}')
print(f'Output: {out.shape}   (should be [4, {EMBEDDING_DIM}])')
print(f'Norms:  {out.norm(dim=1)}  (will NOT be 1.0 — small at init)')

## Cell 5 — Initialize Model + Optimizer

Note: weight_decay=1e-4 (was 0 in notebooks 04/07/07b) — added to prevent embedding-norm explosion since the loss no longer constrains norms via L2 normalization.

In [ ]:
encoder   = CNNEncoderNoL2(embedding_dim=EMBEDDING_DIM).to(device)
optimizer = optim.Adam(encoder.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

print(f'Model on:    {device}')
print(f'Parameters:  {sum(p.numel() for p in encoder.parameters()):,}')
print(f'\nTraining config:')
print(f'  Epochs:        {EPOCHS}')
print(f'  Batch size:    {BATCH_SIZE}')
print(f'  Temperature:   {TEMPERATURE}  (raw dot product InfoNCE)')
print(f'  LR:            {LR} → {1e-5}  (cosine)')
print(f'  Weight decay:  {WEIGHT_DECAY}  (NEW — for stability without L2 norm)')
print(f'  Embedding dim: {EMBEDDING_DIM}  (NO L2 normalization)')
print(f'  Train batches/epoch: {len(train_loader)}')

## Cell 6 — Full Training Loop (with Norm Trajectory Tracking)

New diagnostic: track embedding norm distribution per epoch. If `mean_norm` grows without bound, weight_decay is too weak and we have norm explosion.

In [ ]:
from tqdm.notebook import tqdm

history = {'train_loss': [], 'val_loss': [], 'epoch_time': [],
           'mean_norm': [], 'std_norm': [], 'max_norm': []}

for epoch in range(EPOCHS):
    t0 = time.time()

    # ---- TRAIN ----
    encoder.train()
    train_loss = 0.0
    epoch_norms = []  # collect embedding norms for diagnostic
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]', leave=False)
    for fA, fB in pbar:
        fA = fA.to(device, non_blocking=True)
        fB = fB.to(device, non_blocking=True)
        zA = encoder(fA); zB = encoder(fB)
        # Track norm distribution (no_grad — purely diagnostic)
        with torch.no_grad():
            epoch_norms.extend(zA.norm(dim=1).cpu().numpy().tolist())
            epoch_norms.extend(zB.norm(dim=1).cpu().numpy().tolist())
        loss = InfoNCE_loss_raw(zA, zB, temperature=TEMPERATURE)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'norm': f'{np.mean(epoch_norms[-128:]):.3f}'})
    train_loss /= len(train_loader)

    # ---- VAL ----
    encoder.eval()
    val_loss = 0.0
    with torch.no_grad():
        for fA, fB in val_loader:
            fA = fA.to(device, non_blocking=True)
            fB = fB.to(device, non_blocking=True)
            zA = encoder(fA); zB = encoder(fB)
            val_loss += InfoNCE_loss_raw(zA, zB, temperature=TEMPERATURE).item()
    val_loss /= len(val_loader)
    scheduler.step()

    epoch_norms = np.array(epoch_norms)
    et = time.time() - t0
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['epoch_time'].append(et)
    history['mean_norm'].append(float(epoch_norms.mean()))
    history['std_norm'].append(float(epoch_norms.std()))
    history['max_norm'].append(float(epoch_norms.max()))

    print(f'Epoch {epoch+1:2d}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}  '
          f'norm μ={epoch_norms.mean():.3f} σ={epoch_norms.std():.3f} max={epoch_norms.max():.2f}  '
          f'lr={scheduler.get_last_lr()[0]:.2e}  time={et:.1f}s')

    if (epoch + 1) % 10 == 0:
        ckpt_path = f'{CHECKPOINT_DIR}/encoder_{RUN_TAG}_epoch_{epoch+1}.pth'
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': encoder.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
        }, ckpt_path)
        print(f'  -> Checkpoint saved: {ckpt_path}')

torch.save(encoder.state_dict(), f'{CHECKPOINT_DIR}/encoder_{RUN_TAG}_final.pth')
with open(f'{CHECKPOINT_DIR}/training_history_{RUN_TAG}.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f'\nTraining complete!')
print(f'Total time:       {sum(history["epoch_time"])/60:.1f} min')
print(f'Best val loss:    {min(history["val_loss"]):.4f}  (epoch {history["val_loss"].index(min(history["val_loss"]))+1})')
print(f'Final mean norm:  {history["mean_norm"][-1]:.3f}')
print(f'Final max norm:   {history["max_norm"][-1]:.2f}  (>100 = explosion warning)')

## Cell 7 — Training Curves + Norm Trajectory

Three-panel diagnostic figure.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'],   label='Val',   linewidth=2)
axes[0].axhline(y=np.log(64), color='gray', linestyle='--', alpha=0.5, label='log(64) baseline')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('InfoNCE Loss')
axes[0].set_title(f'Training Curves (τ={TEMPERATURE}, no L2)', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Norm trajectory
axes[1].plot(history['mean_norm'], label='Mean norm', linewidth=2)
axes[1].fill_between(range(len(history['mean_norm'])),
                     np.array(history['mean_norm']) - np.array(history['std_norm']),
                     np.array(history['mean_norm']) + np.array(history['std_norm']),
                     alpha=0.3, label='±1σ')
axes[1].plot(history['max_norm'], label='Max norm', color='red', linewidth=1, linestyle='--')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Embedding norm')
axes[1].set_title('Norm Trajectory (was 1.0 with L2)', fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)

# Epoch time
axes[2].plot(history['epoch_time'], color='green', linewidth=2)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Time (s)')
axes[2].set_title('Training Time per Epoch', fontweight='bold')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Avg epoch time: {np.mean(history["epoch_time"]):.1f}s')
print(f'Total time:     {sum(history["epoch_time"])/60:.1f} min')

# Norm-explosion safety check
if max(history['max_norm']) > 100:
    print('\n⚠️  WARNING: max embedding norm exceeded 100 — possible explosion.')
    print('   Consider: increase WEIGHT_DECAY to 1e-3, or fall back to Variant 1 (cosine sim inside loss).')
else:
    print(f'\n✓ Norms stable (max ever = {max(history["max_norm"]):.2f})')

## Cell 8 — Extract Embeddings + 2D Scatter

Embeddings are NOT L2-normalized, so the scatter scale reflects actual radii. Plot four-panel: by phase, by ENSO, **by amplitude (continuous color)**, and a polar-like angular histogram.

In [ ]:
encoder.eval()
embeddings_2d = np.zeros((len(X), EMBEDDING_DIM), dtype=np.float32)
BATCH = 128
with torch.no_grad():
    for start in range(0, len(X), BATCH):
        end   = min(start + BATCH, len(X))
        batch = torch.from_numpy(X[start:end]).float().to(device)
        embeddings_2d[start:end] = encoder(batch).cpu().numpy()

emb_path = f'{RESULTS_DIR}/embeddings.npy'
np.save(emb_path, embeddings_2d)
print(f'Embeddings shape: {embeddings_2d.shape}')

norms_all = np.linalg.norm(embeddings_2d, axis=1)
angles_all = np.arctan2(embeddings_2d[:, 1], embeddings_2d[:, 0])
print(f'Norm:  min={norms_all.min():.3f}  max={norms_all.max():.3f}  mean={norms_all.mean():.3f}  std={norms_all.std():.3f}')
print(f'Angle: min={angles_all.min():.3f}  max={angles_all.max():.3f}  spread={angles_all.max() - angles_all.min():.3f} rad')

# --- 4-panel scatter ---
Z_val = embeddings_2d[val_idx]
lv = labels.loc[val_idx]

phase_colors = plt.cm.tab10(np.linspace(0, 0.8, 8))
enso_palette = {'El Nino': '#d62728', 'Neutral': '#7f7f7f', 'La Nina': '#1f77b4'}
enso_marker  = {'El Nino': '^',       'Neutral': 'o',        'La Nina': 's'}

# axis range with margin
rng_lo = min(Z_val.min(), -1.1); rng_hi = max(Z_val.max(), 1.1)
pad = 0.1 * (rng_hi - rng_lo)
rng_lo -= pad; rng_hi += pad

fig, axes = plt.subplots(1, 4, figsize=(24, 6))

# (a) by BSISO phase
ax = axes[0]
for ph in range(1, 9):
    m = lv['bsiso_phase'] == ph
    ax.scatter(Z_val[m, 0], Z_val[m, 1], c=[phase_colors[ph-1]], s=18, alpha=0.7, label=f'P{ph}')
ax.set_title('By BSISO Phase', fontweight='bold')
ax.set_xlabel('z₁'); ax.set_ylabel('z₂')
ax.set_aspect('equal'); ax.set_xlim(rng_lo, rng_hi); ax.set_ylim(rng_lo, rng_hi)
ax.axhline(0, color='k', lw=0.4, alpha=0.3); ax.axvline(0, color='k', lw=0.4, alpha=0.3)
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.2)

# (b) by ENSO
ax = axes[1]
for cat in ['El Nino', 'Neutral', 'La Nina']:
    m = lv['enso_category'] == cat
    ax.scatter(Z_val[m, 0], Z_val[m, 1], c=enso_palette[cat], marker=enso_marker[cat], s=18, alpha=0.5, label=cat)
ax.set_title('By ENSO Category', fontweight='bold')
ax.set_xlabel('z₁'); ax.set_ylabel('z₂')
ax.set_aspect('equal'); ax.set_xlim(rng_lo, rng_hi); ax.set_ylim(rng_lo, rng_hi)
ax.axhline(0, color='k', lw=0.4, alpha=0.3); ax.axvline(0, color='k', lw=0.4, alpha=0.3)
ax.legend(); ax.grid(alpha=0.2)

# (c) by BSISO amplitude (continuous)
ax = axes[2]
sc = ax.scatter(Z_val[:, 0], Z_val[:, 1], c=lv['bsiso_amplitude'].values,
                cmap='viridis', s=18, alpha=0.7)
ax.set_title('By BSISO Amplitude', fontweight='bold')
ax.set_xlabel('z₁'); ax.set_ylabel('z₂')
ax.set_aspect('equal'); ax.set_xlim(rng_lo, rng_hi); ax.set_ylim(rng_lo, rng_hi)
ax.axhline(0, color='k', lw=0.4, alpha=0.3); ax.axvline(0, color='k', lw=0.4, alpha=0.3)
plt.colorbar(sc, ax=ax, label='BSISO amplitude'); ax.grid(alpha=0.2)

# (d) angular histogram (val) with phase color overlay
ax = axes[3]
angles_val = np.arctan2(Z_val[:, 1], Z_val[:, 0])
ax.hist(angles_val, bins=36, color='steelblue', alpha=0.7)
ax.set_xlabel('Angle θ (rad)')
ax.set_ylabel('Count')
ax.set_title('Angular Distribution (val)', fontweight='bold')
ax.grid(alpha=0.3)
ax.set_xlim(-np.pi, np.pi)

plt.suptitle(f'Option B (no L2 norm, τ={TEMPERATURE}) — 2D Embedding (val set)',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/embedding_2d_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/lee_2d_no_l2/embedding_2d_overview.png')

## Cell 9 — Radius Diagnostics: Did the Freed Radius Encode Useful Info?

The core test of Option B. We check whether the embedding **radius** correlates with:
- BSISO **amplitude** (continuous regression — the natural hypothesis)
- BSISO **phase** (categorical — radii systematically different across phases?)
- **ENSO** category (does ENSO express as radial offset?)

If radius correlates strongly with amplitude (e.g. Pearson r > 0.5), we have a clean physical interpretation: angle = phase, radius = amplitude, just like (PC1, PC2) but learned end-to-end.

In [ ]:
from scipy.stats import pearsonr, spearmanr, f_oneway

radii  = np.linalg.norm(embeddings_2d, axis=1)
angles = np.arctan2(embeddings_2d[:, 1], embeddings_2d[:, 0])
ampl   = labels['bsiso_amplitude'].values
phase  = labels['bsiso_phase'].values
enso   = labels['enso_category'].values

# --- 1. radius vs BSISO amplitude (continuous) ---
r_pearson,  p_pearson  = pearsonr(radii, ampl)
r_spearman, p_spearman = spearmanr(radii, ampl)
print(f'Radius vs BSISO amplitude:')
print(f'  Pearson  r = {r_pearson:.3f}   p = {p_pearson:.2e}')
print(f'  Spearman r = {r_spearman:.3f}   p = {p_spearman:.2e}')

# --- 2. radius by BSISO phase (categorical) ---
radii_by_phase = [radii[phase == p] for p in range(1, 9)]
f_phase, p_phase = f_oneway(*radii_by_phase)
print(f'\nRadius across BSISO phases (one-way ANOVA): F = {f_phase:.2f}  p = {p_phase:.2e}')

# --- 3. radius by ENSO category ---
radii_by_enso = {c: radii[enso == c] for c in ['El Nino', 'Neutral', 'La Nina']}
f_enso, p_enso = f_oneway(*radii_by_enso.values())
print(f'Radius across ENSO categories (one-way ANOVA): F = {f_enso:.2f}  p = {p_enso:.2e}')

# --- Plot ---
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Panel 1: scatter radius vs amplitude
ax = axes[0]
ax.scatter(ampl, radii, alpha=0.3, s=8)
ax.set_xlabel('BSISO amplitude'); ax.set_ylabel('Embedding radius')
ax.set_title(f'Radius vs Amplitude\nPearson r = {r_pearson:.3f}  (p = {p_pearson:.2e})', fontweight='bold')
ax.grid(alpha=0.3)
# fit line
z_fit = np.polyfit(ampl, radii, 1)
x_fit = np.linspace(ampl.min(), ampl.max(), 100)
ax.plot(x_fit, np.polyval(z_fit, x_fit), 'r-', linewidth=2, label=f'lin fit')
ax.legend()

# Panel 2: boxplot by phase
ax = axes[1]
bp = ax.boxplot(radii_by_phase, positions=range(1, 9), widths=0.6,
                patch_artist=True, showfliers=False)
for patch, color in zip(bp['boxes'], plt.cm.tab10(np.linspace(0, 0.8, 8))):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_xlabel('BSISO phase'); ax.set_ylabel('Embedding radius')
ax.set_title(f'Radius by Phase\nANOVA F = {f_phase:.2f}  (p = {p_phase:.2e})', fontweight='bold')
ax.grid(alpha=0.3)

# Panel 3: boxplot by ENSO
ax = axes[2]
cats = ['El Nino', 'Neutral', 'La Nina']
bp = ax.boxplot([radii_by_enso[c] for c in cats], positions=[1, 2, 3], widths=0.6,
                patch_artist=True, showfliers=False)
for patch, c in zip(bp['boxes'], cats):
    patch.set_facecolor({'El Nino': '#d62728', 'Neutral': '#7f7f7f', 'La Nina': '#1f77b4'}[c])
    patch.set_alpha(0.7)
ax.set_xticks([1, 2, 3]); ax.set_xticklabels(cats)
ax.set_xlabel('ENSO category'); ax.set_ylabel('Embedding radius')
ax.set_title(f'Radius by ENSO\nANOVA F = {f_enso:.2f}  (p = {p_enso:.2e})', fontweight='bold')
ax.grid(alpha=0.3)

plt.suptitle('Option B — Did the Freed Radius Encode Anything?', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/radius_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/lee_2d_no_l2/radius_diagnostics.png')

# Save numerical summary
radius_summary = {
    'radius_vs_amplitude_pearson_r':  float(r_pearson),
    'radius_vs_amplitude_pearson_p':  float(p_pearson),
    'radius_vs_amplitude_spearman_r': float(r_spearman),
    'radius_vs_amplitude_spearman_p': float(p_spearman),
    'radius_by_phase_anova_F': float(f_phase), 'radius_by_phase_anova_p': float(p_phase),
    'radius_by_enso_anova_F':  float(f_enso),  'radius_by_enso_anova_p':  float(p_enso),
    'mean_radius': float(radii.mean()),
    'std_radius':  float(radii.std()),
    'min_radius':  float(radii.min()),
    'max_radius':  float(radii.max()),
}
with open(f'{RESULTS_DIR}/radius_summary.json', 'w') as f:
    json.dump(radius_summary, f, indent=2)
print('Saved: results/lee_2d_no_l2/radius_summary.json')

## Cell 10 — Linear Probes (Full 2D vs Angle-Only)

**The cleanest test of "did the radius help?"**
- **Full 2D probe** uses both (z₁, z₂).
- **Angle-only probe** uses only `θ = atan2(z₂, z₁)` as a single feature (after sin/cos expansion to handle wraparound: `[cos θ, sin θ]`, which is equivalent to L2-normalized 2D).

If full 2D ≈ angle-only → radius added nothing, even when free.  
If full 2D >> angle-only → radius IS encoding useful info.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report
from sklearn.model_selection import cross_val_score, GroupKFold

year_groups = labels['date'].dt.year.values

# Two feature sets
emb_full = embeddings_2d                                       # (N, 2)
emb_angle = np.stack([np.cos(angles), np.sin(angles)], axis=1) # (N, 2) — = L2-normalized version

print('=' * 75)
print('LINEAR PROBES — Full 2D vs Angle-Only  (option B, no L2 norm, τ=0.5)')
print('=' * 75)

results = {}
for feat_name, feat in [('Full 2D (no L2)', emb_full),
                         ('Angle only (L2-normalized equivalent)', emb_angle)]:
    print(f'\n--- Features: {feat_name} ---')
    Z_train = feat[train_idx]; Z_val_ = feat[val_idx]
    gkf = GroupKFold(n_splits=5)

    # BSISO phase
    y_tr = labels.loc[train_idx, 'bsiso_phase'].values
    y_va = labels.loc[val_idx,   'bsiso_phase'].values
    clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    clf.fit(Z_train, y_tr)
    phase_val_acc = float(accuracy_score(y_va, clf.predict(Z_val_)))
    cv = cross_val_score(clf, feat, labels['bsiso_phase'].values,
                          cv=gkf, groups=year_groups, scoring='accuracy', n_jobs=-1)

    # ENSO balanced
    y_tr = labels.loc[train_idx, 'enso_category'].values
    y_va = labels.loc[val_idx,   'enso_category'].values
    clf_bal = LogisticRegression(max_iter=1000, C=1.0, random_state=42, class_weight='balanced')
    clf_bal.fit(Z_train, y_tr)
    enso_bal_val = float(balanced_accuracy_score(y_va, clf_bal.predict(Z_val_)))
    cv_b = cross_val_score(clf_bal, feat, labels['enso_category'].values,
                            cv=gkf, groups=year_groups, scoring='balanced_accuracy', n_jobs=-1)

    print(f'  BSISO phase val:  {phase_val_acc*100:.1f}%   (5-fold CV: {cv.mean()*100:.1f}% ± {cv.std()*100:.1f}%)')
    print(f'  ENSO bal-acc val: {enso_bal_val*100:.1f}%   (5-fold CV: {cv_b.mean()*100:.1f}% ± {cv_b.std()*100:.1f}%)')

    results[feat_name] = {
        'phase_val_acc': phase_val_acc,
        'phase_cv_mean': float(cv.mean()), 'phase_cv_std': float(cv.std()),
        'enso_bal_val':  enso_bal_val,
        'enso_cv_mean':  float(cv_b.mean()), 'enso_cv_std':  float(cv_b.std()),
    }

with open(f'{RESULTS_DIR}/angle_only_vs_full_probes.json', 'w') as f:
    json.dump(results, f, indent=2)

# Standard probe results file (for parity with earlier notebooks)
with open(f'{RESULTS_DIR}/linear_probe_results.json', 'w') as f:
    json.dump({
        'BSISO Phase': {
            'val_acc':  results['Full 2D (no L2)']['phase_val_acc'],
            'cv_mean':  results['Full 2D (no L2)']['phase_cv_mean'],
            'cv_std':   results['Full 2D (no L2)']['phase_cv_std'],
        },
        'ENSO Category (balanced)': {
            'val_bal_acc': results['Full 2D (no L2)']['enso_bal_val'],
            'cv_mean':     results['Full 2D (no L2)']['enso_cv_mean'],
            'cv_std':      results['Full 2D (no L2)']['enso_cv_std'],
        },
    }, f, indent=2)

print('\nSaved: results/lee_2d_no_l2/{linear_probe_results.json, angle_only_vs_full_probes.json}')

# Quick verdict on radius contribution
gap_phase = (results['Full 2D (no L2)']['phase_val_acc'] -
             results['Angle only (L2-normalized equivalent)']['phase_val_acc']) * 100
print(f'\nRadius contribution: full 2D − angle-only = {gap_phase:+.1f} pp on BSISO phase val acc')

## Cell 11 — ENSO Displacement Z-Score

Same statistic as notebook 05 / 07 / 07b, computed in the unnormalized 2D space.

In [ ]:
phases = range(1, 9)
disp_mag = []
for ph in phases:
    mEN = (labels['bsiso_phase'] == ph) & (labels['enso_category'] == 'El Nino')
    mLN = (labels['bsiso_phase'] == ph) & (labels['enso_category'] == 'La Nina')
    if mEN.sum() < 3 or mLN.sum() < 3:
        disp_mag.append(np.nan); continue
    cEN = embeddings_2d[mEN].mean(axis=0); cLN = embeddings_2d[mLN].mean(axis=0)
    disp_mag.append(np.linalg.norm(cEN - cLN))

rng = np.random.default_rng(42)
baseline_mag = []
for _ in range(100):
    shuf = labels['enso_category'].sample(frac=1, random_state=rng.integers(1e6)).values
    mtrl = []
    for ph in phases:
        mph = (labels['bsiso_phase'] == ph).values
        mEN = mph & (shuf == 'El Nino'); mLN = mph & (shuf == 'La Nina')
        if mEN.sum() < 3 or mLN.sum() < 3: continue
        mtrl.append(np.linalg.norm(embeddings_2d[mEN].mean(axis=0) - embeddings_2d[mLN].mean(axis=0)))
    if mtrl: baseline_mag.append(np.mean(mtrl))

bmu = float(np.mean(baseline_mag))
bsd = float(np.std(baseline_mag))
obs_mu = float(np.nanmean(disp_mag))
z_score = float((obs_mu - bmu) / (bsd + 1e-8))

print(f'EN–LN displacement summary (Option B, no L2):')
print(f'  Observed mean: {obs_mu:.4f}')
print(f'  Null baseline: {bmu:.4f} ± {bsd:.4f}')
print(f'  Z-score:       {z_score:.2f}   (64D baseline: 3.83;  L2-norm best τ=0.5: 2.59)')

fig, ax = plt.subplots(figsize=(8, 5))
valid_phases = [p for p, m in zip(phases, disp_mag) if not np.isnan(m)]
valid_mags = [m for m in disp_mag if not np.isnan(m)]
ax.bar(valid_phases, valid_mags, color='steelblue', alpha=0.8)
ax.axhline(bmu, color='red', linestyle='--', linewidth=1.5, label=f'Null mean ({bmu:.3f})')
ax.axhline(bmu + 2*bsd, color='red', linestyle=':', linewidth=1, label='Null +2σ')
ax.axhline(obs_mu, color='steelblue', linewidth=2, label=f'Observed mean ({obs_mu:.3f})')
ax.set_xticks(range(1, 9))
ax.set_xlabel('BSISO Phase'); ax.set_ylabel('||EN−LN||')
ax.set_title(f'ENSO Displacement (Option B), z = {z_score:.2f}  (64D baseline: 3.83)', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/enso_displacement.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/lee_2d_no_l2/enso_displacement.png')

## Cell 12 — Comparison Table + Auto-Decision

Summarises:
- 64D baseline
- 2D L2-norm best (τ=0.5 from 07b sweep)
- 2D Option B (this notebook)

plus the angle-only-vs-full-2D probe to verify whether the radius contributes.

In [ ]:
# Hardcoded baselines from prior runs (single source of truth — update if reruns happen)
BASELINE_64D = {'phase_val_acc': 0.677, 'phase_cv': '68.6% ± 1.0%', 'z_score': 3.83}
BASELINE_2D_L2_TAU05 = {'phase_val_acc': 0.332, 'phase_cv': '38.2% ± 3.5%', 'z_score': 2.59,
                        'angular_spread_rad': 6.28}

opt_b = {
    'phase_val_acc': results['Full 2D (no L2)']['phase_val_acc'],
    'phase_cv':      f"{results['Full 2D (no L2)']['phase_cv_mean']*100:.1f}% ± {results['Full 2D (no L2)']['phase_cv_std']*100:.1f}%",
    'enso_bal_val':  results['Full 2D (no L2)']['enso_bal_val'],
    'z_score':       z_score,
    'mean_radius':   float(radii.mean()),
    'std_radius':    float(radii.std()),
    'r_radius_amplitude': float(r_pearson),
}

rows = [
    ['64D baseline (Lee MJJAS)',            f"{BASELINE_64D['phase_val_acc']*100:.1f}%", BASELINE_64D['phase_cv'],         'n/a',                                                       f"{BASELINE_64D['z_score']:.2f}",         'n/a',          'n/a (on S⁶³)'],
    ['2D L2-norm best τ=0.5 (notebook 07b)', f"{BASELINE_2D_L2_TAU05['phase_val_acc']*100:.1f}%", BASELINE_2D_L2_TAU05['phase_cv'],  'n/a (used standard acc)',                                  f"{BASELINE_2D_L2_TAU05['z_score']:.2f}", 'n/a (=1)',     '—'],
    ['2D Option B (no L2, τ=0.5) — this',   f"{opt_b['phase_val_acc']*100:.1f}%",            opt_b['phase_cv'],                  f"{opt_b['enso_bal_val']*100:.1f}%",                       f"{opt_b['z_score']:.2f}",            f"{opt_b['mean_radius']:.2f} ± {opt_b['std_radius']:.2f}", f"r={opt_b['r_radius_amplitude']:.2f}"],
]
df_comp = pd.DataFrame(rows, columns=[
    'Configuration', 'BSISO phase val', 'BSISO 5-fold CV',
    'ENSO bal-acc val', 'z-score', 'mean radius', 'radius~amplitude r'
])

print('=' * 110)
print('OPTION B vs PRIOR RESULTS')
print('=' * 110)
print(df_comp.to_string(index=False))

# Angle-only vs full 2D — does the radius contribute?
print('\n' + '=' * 110)
print('IS THE RADIUS USEFUL? — Angle-only vs Full-2D probe (both probe Option B embeddings)')
print('=' * 110)
for feat_name, r in results.items():
    print(f"  {feat_name}:  phase val = {r['phase_val_acc']*100:.1f}%   ENSO bal = {r['enso_bal_val']*100:.1f}%")
gap_phase = (results['Full 2D (no L2)']['phase_val_acc'] - results['Angle only (L2-normalized equivalent)']['phase_val_acc']) * 100
gap_enso  = (results['Full 2D (no L2)']['enso_bal_val']  - results['Angle only (L2-normalized equivalent)']['enso_bal_val'])  * 100
print(f'\nGap (full − angle-only):  phase = {gap_phase:+.1f} pp   ENSO = {gap_enso:+.1f} pp')
print('Interpretation: positive gap = radius contributes; near zero = radius is unused.')

# --- Decision ---
phase_pass = opt_b['phase_val_acc'] >= 0.62
z_pass     = opt_b['z_score']      >= 3.0
radius_useful = abs(gap_phase) >= 5.0   # ≥ 5 pp gap = noticeable contribution
norm_explosion = max(history['max_norm']) > 100

if norm_explosion:
    decision = 'NORM_EXPLOSION'
    decision_text = (f"Embedding norms exploded (max = {max(history['max_norm']):.1f}). Variant 2 unstable. "
                     f"Recommend: rerun with WEIGHT_DECAY=1e-3, or fall back to Variant 1 (cosine sim inside loss).")
elif phase_pass and z_pass:
    decision = 'GREENLIGHT_PHASE_2'
    decision_text = (f"**BSISO is genuinely 2D in R² without L2 norm.** Phase val {opt_b['phase_val_acc']*100:.1f}% "
                     f"(≥ 62% threshold) and z={opt_b['z_score']:.2f} (≥ 3.0 threshold). "
                     f"Use this config (no L2, τ=0.5, weight_decay=1e-4) for **Phase 2 SSL temporal model**.")
elif radius_useful and (opt_b['phase_val_acc'] > 0.45 or opt_b['z_score'] > 2.5):
    decision = 'PARTIAL_IMPROVEMENT'
    decision_text = (f"**Radius helps but doesn't fully recover.** Phase val {opt_b['phase_val_acc']*100:.1f}% (vs 33.2% L2-norm), "
                     f"z={opt_b['z_score']:.2f}, full-vs-angle gap = {gap_phase:+.1f} pp. "
                     f"Below the 64D baseline (67.7%, z=3.83) but above the 1D-circle ceiling. "
                     f"→ Discuss with advisor: Phase 2 with this setup vs proceed to Phase 4 dim sweep.")
else:
    decision = 'RADIUS_DID_NOT_HELP'
    decision_text = (f"**The freed radius added little.** Full-2D and angle-only probes are within {gap_phase:.1f} pp; "
                     f"phase val {opt_b['phase_val_acc']*100:.1f}% is similar to the L2-norm result. "
                     f"BSISO under standard contrastive learning genuinely doesn't fit in 2D R². "
                     f"→ **Escalate to Phase 4 dimension sweep** {{1,2,4,8,16,32,64}}.")

print('\n' + '=' * 110)
print(f'DECISION: {decision}')
print('=' * 110)
print(decision_text)

# --- Summary markdown ---
import time as _t
summary = f"""# Phase 1 — Option B (no L2 normalization) Summary

**Auto-generated by notebook 07c.**  
**Date:** {_t.strftime('%Y-%m-%d')}  
**Model:** `encoder_{RUN_TAG}_final.pth`, embedding_dim=2, no L2 norm, τ={TEMPERATURE}, weight_decay={WEIGHT_DECAY}

## Comparison

{df_comp.to_markdown(index=False)}

**64D baseline:** Lee MJJAS, year-based split, phase val 67.7%, ENSO z 3.83.  
**2D L2-norm best:** τ=0.5 from notebook 07b temperature sweep — phase val 33.2%, z 2.59, full angular spread.

## Did the radius help?

| Probe features | BSISO phase val | ENSO bal-acc val |
|---|---|---|
| Full 2D (no L2) | {results['Full 2D (no L2)']['phase_val_acc']*100:.1f}% | {results['Full 2D (no L2)']['enso_bal_val']*100:.1f}% |
| Angle-only (L2-normalized equivalent) | {results['Angle only (L2-normalized equivalent)']['phase_val_acc']*100:.1f}% | {results['Angle only (L2-normalized equivalent)']['enso_bal_val']*100:.1f}% |

**Gap (full − angle-only):** phase = {gap_phase:+.1f} pp, ENSO = {gap_enso:+.1f} pp.

## Radius diagnostics

- Pearson(radius, BSISO amplitude) = **{r_pearson:.3f}** (p = {p_pearson:.2e})
- Radius across phases: ANOVA F = {f_phase:.2f}, p = {p_phase:.2e}
- Radius across ENSO: ANOVA F = {f_enso:.2f}, p = {p_enso:.2e}
- Mean radius: {radii.mean():.3f} ± {radii.std():.3f}  (was 1.0 with L2 norm)
- Max norm during training: {max(history['max_norm']):.2f}  (>100 = explosion)

## Decision

{decision_text}

## Files produced

```
checkpoints/encoder_{RUN_TAG}_final.pth
checkpoints/training_history_{RUN_TAG}.json
results/lee_2d_no_l2/embeddings.npy
results/lee_2d_no_l2/training_curves.png
results/lee_2d_no_l2/embedding_2d_overview.png
results/lee_2d_no_l2/radius_diagnostics.png
results/lee_2d_no_l2/radius_summary.json
results/lee_2d_no_l2/linear_probe_results.json
results/lee_2d_no_l2/angle_only_vs_full_probes.json
results/lee_2d_no_l2/enso_displacement.png
results/lee_2d_no_l2/phase1_option_b_summary.md   (this file)
```
"""

with open(f'{RESULTS_DIR}/phase1_option_b_summary.md', 'w') as f:
    f.write(summary)
print(f'\nSaved: {RESULTS_DIR}/phase1_option_b_summary.md')

## Cell 13 — (Optional) Download All Outputs to Local Machine

In [ ]:
from google.colab import files
for fname in sorted(os.listdir(RESULTS_DIR)):
    files.download(f'{RESULTS_DIR}/{fname}')

---
## Done!

**Send back to me:**
1. `phase1_option_b_summary.md` (the auto-decision)
2. `embedding_2d_overview.png` (4-panel scatter, including by-amplitude)
3. `radius_diagnostics.png` (the key new figure)
4. `training_curves.png` (to check norms didn't explode)

I'll log the decision into the conversation log and the plan checklist, and depending on the verdict either:
- start `08_ssl_temporal_2d.ipynb` (Phase 2) with the Option B config, or
- write `07d_dim_sweep.ipynb` (Phase 4) for the dimension sweep.

---
*DDCS Project | jh9141@nyu.edu*